# 04 — Imagens como dados: cutouts, RGB e dataset visual

Workshop SAB — AI-Scope / ADSS

**Pergunta científica:** como transformar uma lista de coordenadas em um conjunto de imagens reproduzível para inspeção ou deep learning?

Aqui a unidade de análise deixa de ser uma linha de catálogo. Usaremos as coleções de arquivos do ADSS para produzir cutouts sob demanda e composições RGB, sem manter cópias locais de campos completos.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adss

BASE_URL = "https://ai-scope.cbpf.br/"
USERNAME = os.getenv("AI_SCOPE_USERNAME")
PASSWORD = os.getenv("AI_SCOPE_PASSWORD")

if not USERNAME or not PASSWORD:
    raise RuntimeError("Defina AI_SCOPE_USERNAME e AI_SCOPE_PASSWORD no ambiente.")

cl = adss.ADSSClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)

In [ ]:
collections = pd.DataFrame(cl.get_image_collections())
collections[[c for c in ["id","name","description"] if c in collections.columns]]

In [ ]:
# Escolha uma coleção que possua WCS e ajuste estas coordenadas a uma região coberta.
COLLECTION_ID = int(collections.iloc[0].id)
RA, DEC = 0.1, 0.1

files = cl.list_files(COLLECTION_ID)
pd.DataFrame(files).head()

In [ ]:
from io import BytesIO
from astropy.io import fits

cutout_bytes = cl.create_stamp_by_coordinates(
    collection_id=COLLECTION_ID,
    ra=RA,
    dec=DEC,
    size=128,
    filter="R",
    size_unit="pixels",
    format="fits",
    pattern="swp.",
)

with fits.open(BytesIO(cutout_bytes)) as hdul:
    image = hdul[0].data if hdul[0].data is not None else hdul[1].data

fig, ax = plt.subplots(figsize=(5,5))
lo, hi = np.nanpercentile(image, [5, 99.5])
ax.imshow(image, origin="lower", vmin=lo, vmax=hi)
ax.set(title="Server-side cutout")
plt.show()

In [ ]:
from PIL import Image

rgb_bytes = cl.create_rgb_image_by_coordinates(
    collection_id=COLLECTION_ID,
    ra=RA,
    dec=DEC,
    size=256,
    size_unit="pixels",
    r_filter="I",
    g_filter="R",
    b_filter="G",
)
rgb = Image.open(BytesIO(rgb_bytes))
rgb

### De visualização para ML
O mesmo mecanismo pode receber uma tabela com milhares de coordenadas e gerar amostras homogêneas. Uma atividade útil é definir um *manifest* com `object_id`, coordenadas, filtros, tamanho do stamp e rótulo científico. Isso torna a construção do dataset rastreável e repetível.